# Banking regression: predict a continuous loan amount
**Target:** `loan_amount_inr`. Unlike classification, regression predicts a number of rupees. We reuse the loan-default CSV; remove `loan_default` because it describes a later outcome.
# Installation and data
Use a Python environment supported by your installed PyCaret release; check the release installation page before installing. In a fresh Python 3.11 environment, run `%pip install pycaret` and restart the kernel. Keep the named CSV alongside this notebook. These notebooks use PyCaret 3 function APIs and the data is for teaching only.


## 1. Read and inspect data
Check dimensions, examples, missing values and the numeric target distribution.

In [ ]:
import pandas as pd
df = pd.read_csv("Banking_Loan_Default_Classification.csv")
print(df.shape)
display(df.head())
display(df.isna().sum().to_frame("missing"))
display(df["loan_amount_inr"].describe())

## 2. Prepare the experiment
PyCaret handles categorical encoding, missing values and splitting. `ignore_features` excludes the post-loan default label to avoid leakage.

In [ ]:
from pycaret.regression import setup, models, compare_models, pull, tune_model, plot_model, predict_model, finalize_model, save_model, load_model
exp = setup(data=df, target="loan_amount_inr", ignore_features=["loan_default"], train_size=0.8, fold=5, session_id=42, verbose=False)
display(models().head())

## 3. Compare models
MAE is the average absolute rupee error; RMSE penalizes large errors more; R² measures explained variation and can be negative on poor holdout models. Lower MAE is preferable.

In [ ]:
best = compare_models(include=["lr", "dt", "rf", "et"], sort="MAE")
display(pull())

## 4. Tune and inspect
Tuning can help, but improvement is not guaranteed. `choose_better=True` returns the better of the original and tuned versions by MAE.

In [ ]:
tuned = tune_model(best, optimize="MAE", choose_better=True, n_iter=10)
display(pull())
plot_model(tuned, plot="residuals")
plot_model(tuned, plot="error")

## 5. Holdout evaluation
The holdout was kept out of model fitting. Compare actual amounts with `prediction_label`; `pull()` shows holdout metrics.

In [ ]:
holdout = predict_model(tuned)
display(holdout[["loan_amount_inr", "prediction_label"]].head(10))
display(pull())

## 6. Save, reload and predict
Finalize refits on all supplied rows after holdout evaluation. New records must have the same input columns.

In [ ]:
final = finalize_model(tuned)
save_model(final, "banking_loan_amount_regression")
loaded = load_model("banking_loan_amount_regression")
new_rows = df.drop(columns=["loan_amount_inr", "loan_default"]).head(3)
display(predict_model(loaded, data=new_rows))